Creating a simple weather heatmap of the US then compare with agricultural commodity futures prices % change today

In [2]:
import yfinance as yf
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt

In [3]:
corn_belt = {
    "Iowa": (41.878, -93.098),
    "Illinois": (40.349, -88.986),
    "Indiana": (40.267, -86.135),
    "Nebraska": (41.493, -99.901),
    "Minnesota": (45.694, -93.900),
}

In [4]:
import requests
import pandas as pd

def get_last_7_days(lat, lon):
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": lat,
        "longitude": lon,
        "daily": "temperature_2m_max,precipitation_sum",
        "past_days": 7,
        "forecast_days": 0,   # we only want the past, not the forecast
        "timezone": "auto",
    }
    response = requests.get(url, params=params)
    data = response.json()["daily"]
    return pd.DataFrame(data)

In [5]:
avg_temps = []
total_rains = []

for state, (lat, lon) in corn_belt.items():
    df = get_last_7_days(lat, lon)
    avg_temps.append(df["temperature_2m_max"].mean())
    total_rains.append(df["precipitation_sum"].sum())

corn_belt_avg_temp_c = sum(avg_temps) / len(avg_temps)
corn_belt_avg_temp_f = corn_belt_avg_temp_c * 9/5 + 32
corn_belt_total_rain_mm = sum(total_rains) / len(total_rains)

print(f"Corn Belt avg high (last 7 days): {corn_belt_avg_temp_f:.1f}°F")
print(f"Corn Belt avg total rainfall (last 7 days): {corn_belt_total_rain_mm:.1f}mm")

Corn Belt avg high (last 7 days): 87.1°F
Corn Belt avg total rainfall (last 7 days): 74.2mm


In [6]:
# simple, adjustable thresholds
is_hot = corn_belt_avg_temp_f > 88
is_dry = corn_belt_total_rain_mm < 15  # ~0.6 inches over a week is low

if is_hot and is_dry:
    print("Corn Belt is HOT and DRY -- potential yield stress signal")
elif is_hot:
    print("Corn Belt is hot but rainfall is adequate")
elif is_dry:
    print("Corn Belt is dry but temps are not extreme")
else:
    print("No significant heat/dryness signal")

No significant heat/dryness signal


In [7]:
import yfinance as yf

tickers = {"ZC=F": "Corn", "ZS=F": "Soybeans", "ZW=F": "Wheat"}

for ticker, name in tickers.items():
    info = yf.Ticker(ticker).fast_info
    change_pct = (info["lastPrice"] / info["previousClose"] - 1) * 100
    print(f"{name}: ${info['lastPrice']:.2f}  ({change_pct:+.2f}%)")

Corn: $449.75  (+0.22%)
Soybeans: $1171.25  (+0.32%)
Wheat: $606.50  (+0.37%)


In [8]:
print(f"\nSUMMARY: Corn Belt {'HOT+DRY' if is_hot and is_dry else 'not stressed'} "
      f"over last 7 days. Corn today: {change_pct:+.2f}%")


SUMMARY: Corn Belt not stressed over last 7 days. Corn today: +0.37%
